# SQL Basics: SELECT, WHERE and ORDER BY
**Goal:** write your first SQL queries and answer five questions about Nkwen Traders' four branches (Chapter 2 data, January to June 2026).

You will use **SQLite**, a small database that comes built into Python. Nothing to install.

**What to submit:** the GitHub commit link to this notebook (File > Save a copy in GitHub), and in the text box your five answers with the SQL query for each.

In [ ]:
import sqlite3
import pandas as pd

SALES_URL = "https://innovakickstarter.com/course-files/data-science/data/nkwen_traders_ch2_sales.csv"
BRANCHES_URL = "https://innovakickstarter.com/course-files/data-science/data/nkwen_traders_branches.csv"

# 1. Load the two CSV files into pandas (Colab downloads them, not your phone or laptop)
sales = pd.read_csv(SALES_URL)
branches = pd.read_csv(BRANCHES_URL)

# 2. Create a small SQLite database file and copy both DataFrames into it as tables
conn = sqlite3.connect("nkwen.db")
sales.to_sql("sales", conn, if_exists="replace", index=False)
branches.to_sql("branches", conn, if_exists="replace", index=False)

# 3. A helper: run a SQL query and get the result back as a DataFrame
def run_sql(query):
    return pd.read_sql_query(query, conn)

print("SQLite version:", sqlite3.sqlite_version)
print("Tables:", run_sql("SELECT name FROM sqlite_master WHERE type = 'table'")["name"].tolist())

### Self-check 1: are the tables loaded?
Run this cell. It checks that the database tables have the same number of rows as the CSV files.

In [ ]:
n_sales_table = run_sql("SELECT COUNT(*) AS n FROM sales")["n"][0]
n_branch_table = run_sql("SELECT COUNT(*) AS n FROM branches")["n"][0]
assert n_sales_table == len(sales), "The sales table is not complete. Re-run the setup cell."
assert n_branch_table == len(branches), "The branches table is not complete. Re-run the setup cell."
print(f"Success: sales has {n_sales_table} rows and branches has {n_branch_table} rows.")

## Step 1: Warm-up - look at the data with SQL
A query always starts with `SELECT ... FROM ...`. `LIMIT` keeps only the first rows.

```sql
SELECT * FROM sales LIMIT 5;
```
Run the example, then write the TODO queries.

In [ ]:
run_sql("SELECT * FROM sales LIMIT 5")

In [ ]:
# TODO: list the columns of the sales table and their types.
# Hint: run_sql("PRAGMA table_info(sales)")

# TODO: show every row of the branches table (it is small).

# TODO: list each branch name ONCE from the sales table (use SELECT DISTINCT).

## Step 2: Filter rows with WHERE
- Text goes in **single quotes**: `WHERE Branch = 'Mile 4'`. SQLite compares text exactly, so `'mile 4'` would match nothing.
- Numbers have no quotes: `WHERE TotalSale_FCFA > 50000` (no comma or space inside the number).
- Dates in this table are text like `2026-03-15`. Because the format is year-month-day, text comparison works:
  `WHERE Date BETWEEN '2026-03-01' AND '2026-03-31'` (BETWEEN includes both ends).
- Combine conditions with `AND` / `OR`. When you mix them, use brackets.
- `IN` is a short way to write several ORs: `WHERE PaymentMethod IN ('Cash', 'Bank Transfer')`.
- `LIKE` matches patterns: `WHERE Product LIKE 'Rice%'` means "starts with Rice".

**Practice (not graded):** count the Cash transactions at Commercial Avenue.

In [ ]:
# Example: count rows with COUNT(*) and give the result a name with AS
run_sql("""
    SELECT COUNT(*) AS transactions
    FROM sales
    WHERE Branch = 'Commercial Avenue'
      AND PaymentMethod = 'Cash'
""")

### Q1. How many Nkwen transactions were above 50,000 FCFA?
Store your query text in `q1_sql`, then run it.

In [ ]:
q1_sql = """
-- TODO: write your query here
"""
r1 = run_sql(q1_sql)
r1

### Q2. How many transactions did Mile 4 record in March 2026?
Make sure your date filter covers the whole month, from the 1st to the 31st.

In [ ]:
q2_sql = """
-- TODO: write your query here
"""
r2 = run_sql(q2_sql)
r2

## Step 3: Sort with ORDER BY and keep the top rows with LIMIT
`ORDER BY TotalSale_FCFA DESC` sorts from biggest to smallest (`ASC`, the default, is smallest first).

### Q3. What were the 5 biggest single sales across all branches?
Show `TransactionID, Date, Branch, Product, Quantity, TotalSale_FCFA`, biggest first.

In [ ]:
q3_sql = """
-- TODO: write your query here
"""
r3 = run_sql(q3_sql)
r3

### Self-check 2: is Q3 a proper top 5?

In [ ]:
assert len(r3) == 5, "Q3 should return exactly 5 rows. Did you use LIMIT 5?"
assert "TotalSale_FCFA" in r3.columns, "Q3 should include the TotalSale_FCFA column."
assert r3["TotalSale_FCFA"].is_monotonic_decreasing, "Q3 should be sorted biggest first. Did you use ORDER BY ... DESC?"
print("Success: Q3 has 5 rows, sorted from the biggest sale down.")

### Q4. Which products belong to the Oils category?
List each product **once**, in alphabetical order. Hint: `SELECT DISTINCT ... ORDER BY ...`.

In [ ]:
q4_sql = """
-- TODO: write your query here
"""
r4 = run_sql(q4_sql)
r4

### Self-check 3: is each product listed once?

In [ ]:
assert len(r4) > 0, "Q4 returned no rows. Check the spelling and capital letter of 'Oils'."
first_col = r4.columns[0]
assert not r4[first_col].duplicated().any(), "A product appears more than once. Did you use SELECT DISTINCT?"
assert r4[first_col].tolist() == sorted(r4[first_col].tolist()), "The list is not in alphabetical order. Add ORDER BY."
print(f"Success: {len(r4)} distinct products, in alphabetical order.")

### Q5. How much revenue (FCFA) did Bafoussam Centre take through mobile money?
Mobile money here means **Mobile Money** or **Orange Money**. Use `SUM(...)` and `IN (...)`.

In [ ]:
q5_sql = """
-- TODO: write your query here
"""
r5 = run_sql(q5_sql)
r5

### Self-check 4: does Q5 make sense?
It checks that you used `IN`, that the result is one number, and that it is smaller than the branch's total revenue.

In [ ]:
import re
assert re.search(r"\bIN\s*\(", q5_sql, flags=re.IGNORECASE), "Use IN ('Mobile Money', 'Orange Money') in your WHERE clause."
assert r5.shape == (1, 1), "Q5 should return one row with one number (the total)."
branch_total = run_sql("SELECT SUM(TotalSale_FCFA) AS t FROM sales WHERE Branch = 'Bafoussam Centre'")["t"][0]
your_value = r5.iloc[0, 0]
assert your_value is not None and 0 < your_value < branch_total, "Your total should be above 0 and below the branch's total revenue. Check the branch name and payment names."
print("Success: Q5 is a single total, and it is a part of Bafoussam Centre's revenue.")

## Step 4: SQL and pandas side by side
Pick **one** of Q1 to Q5 and answer it again with pandas (the `sales` DataFrame is already loaded).
Check that both answers are the same.

In [ ]:
# TODO: pandas version of one question, for example:
# sales[(sales["Branch"] == "...") & (sales["TotalSale_FCFA"] > ...)].shape[0]

**Your reflection (TODO, 1-2 sentences):** which version was clearer for you, SQL or pandas, and why?

### Before you submit
- [ ] **Runtime > Restart session and run all**: every cell runs from top to bottom without an error.
- [ ] Every self-check cell printed its success message.
- [ ] Your answers are written in Markdown cells in your own words, not only as code output.
- [ ] Save to GitHub: **File > Save a copy in GitHub**, choose your repository, write a clear commit message, and click OK.
- [ ] Open the notebook on GitHub, check that the outputs show, and copy the commit link for your submission.